In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# -------------------------------
# 1. LOAD DATASET
# -------------------------------
data = pd.read_csv("smart_dryer_dataset.csv").dropna()

X = data[['Temperature', 'Humidity', 'Time', 'Moisture']]
y_fan = data['Fan_Index']
y_heater = data['Heater']
y_warning = data['Warning']

# -------------------------------
# 2. PREPROCESSING
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, yf_train, yf_test, yh_train, yh_test, yw_train, yw_test = train_test_split(
    X_scaled, y_fan, y_heater, y_warning, test_size=0.2, random_state=42
)

# -------------------------------
# 3. BUILD MODEL
# -------------------------------
inputs = tf.keras.Input(shape=(4,))

x = tf.keras.layers.Dense(64, activation='relu')(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Dense(32, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)

fan_output = tf.keras.layers.Dense(1, activation='relu', name='fan')(x)
heater_output = tf.keras.layers.Dense(1, activation='sigmoid', name='heater')(x)
warning_output = tf.keras.layers.Dense(1, activation='sigmoid', name='warning')(x)

model = tf.keras.Model(inputs=inputs, outputs=[fan_output, heater_output, warning_output])

# -------------------------------
# 4. COMPILE
# -------------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={
        'fan': 'mse',
        'heater': 'binary_crossentropy',
        'warning': 'binary_crossentropy'
    },
    loss_weights={
        'fan': 1.0,
        'heater': 1.0,
        'warning': 2.0
    },
    metrics={
        'fan': 'mae',
        'heater': 'accuracy',
        'warning': 'accuracy'
    }
)

# -------------------------------
# 5. TRAIN
# -------------------------------
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    {'fan': yf_train, 'heater': yh_train, 'warning': yw_train},
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop]
)

# -------------------------------
# 6. EVALUATE
# -------------------------------
model.evaluate(
    X_test,
    {'fan': yf_test, 'heater': yh_test, 'warning': yw_test}
)

# -------------------------------
# 7. PLOT LOSS
# -------------------------------
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Loss Curve")
plt.show()

# -------------------------------
# 8. EXPLAINABLE PREDICTION
# -------------------------------
def predict_system(temp, hum, time, moist):

    input_data = np.array([[temp, hum, time, moist]])
    input_scaled = scaler.transform(input_data)

    fan, heater, warning = model.predict(input_scaled)

    fan_value = float(fan[0][0])
    heater_prob = float(heater[0][0])
    warning_prob = float(warning[0][0])

    heater_status = 1 if heater_prob > 0.5 else 0
    warning_status = 1 if warning_prob > 0.5 else 0

    # -------------------------------
    # SAFETY RULES
    # -------------------------------
    if temp > 60:
        heater_status = 0
        warning_status = 1

    if moist < 10:
        heater_status = 0

    # -------------------------------
    # FAN INTERPRETATION
    # -------------------------------
    if fan_value < 30:
        fan_level = "LOW"
    elif fan_value < 70:
        fan_level = "MEDIUM"
    else:
        fan_level = "HIGH"

    # -------------------------------
    # EXPLANATION ENGINE (KEY)
    # -------------------------------
    reasons = []

    if moist > 60:
        reasons.append("High moisture → strong drying required")

    if hum > 70:
        reasons.append("High humidity → slows drying → increase airflow")

    if temp > 50:
        reasons.append("High temperature → reduce heating for safety")

    if moist < 20:
        reasons.append("Low moisture → drying nearly complete")

    if temp > 60:
        reasons.append("Overheat condition → system safety activated")

    if warning_status == 1:
        reasons.append("Warning triggered due to unsafe conditions")

    # -------------------------------
    # OUTPUT DISPLAY
    # -------------------------------
    print("\n===== SMART DRYER OUTPUT =====")
    print(f"Input Conditions → Temp: {temp}°C | Humidity: {hum}% | Moisture: {moist}%")

    print("\nPredicted Actions:")
    print(f"Airflow Level : {fan_level}")
    print(f"Heater        : {'ON' if heater_status else 'OFF'}")
    print(f"Warning       : {'ALERT' if warning_status else 'SAFE'}")

    print("\nExplanation:")
    if reasons:
        for r in reasons:
            print("•", r)
    else:
        print("• Conditions normal, system operating optimally")

# -------------------------------
# 9. TEST
# -------------------------------
predict_system(35, 75, 20, 80)

FileNotFoundError: [Errno 2] No such file or directory: 'smart_dryer_dataset.csv'